In [0]:
from __future__ import annotations

import sys
from pathlib import Path

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DateType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

# Databricks Repos normally adds the repository root to sys.path.
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "notebooks" / "common").exists():
        repository_root = str(candidate)
        if repository_root not in sys.path:
            sys.path.insert(0, repository_root)
        break

from notebooks.common.audit import add_audit_columns
from notebooks.common.metrics import (
    calculate_data_quality_score,
    create_pipeline_metric_df,
    print_summary,
)
from notebooks.common.paths import PATHS

import importlib
import notebooks.common.audit as audit_module

importlib.reload(audit_module)
add_audit_columns = audit_module.add_audit_columns

In [0]:
NOTEBOOK_VERSION = "1.0.0"
PIPELINE_NAME = "northstar_bronze_to_silver_enrollments"

BUSINESS_DATE = "2026-07-01"
BATCH_ID = "northstar-enrollments-20260701"
SOURCE_SYSTEM = "Enrollment Platform"

WRITE_MODE = "overwrite"

BRONZE_PATH = (
    f"abfss://bronze@{PATHS.storage_account}.dfs.core.windows.net/"
    "northstar/enrollment/enrollments/enrollments_20260701.csv"
)

SILVER_PATH = PATHS.enrollments_silver

QUARANTINE_PATH = (
    f"abfss://gold@{PATHS.storage_account}.dfs.core.windows.net/"
    "northstar/quarantine/enrollments"
)

METRICS_PATH = (
    f"abfss://gold@{PATHS.storage_account}.dfs.core.windows.net/"
    "northstar/data_quality_metrics"
)

print(BRONZE_PATH)
print(SILVER_PATH)

In [0]:
enrollment_schema = StructType([
    StructField("enrollment_id", StringType(), True),
    StructField("employer_id", StringType(), True),
    StructField("employee_id", StringType(), True),
    StructField("dependent_id", StringType(), True),
    StructField("plan_id", StringType(), True),
    StructField("coverage_start_date", DateType(), True),
    StructField("coverage_end_date", DateType(), True),
    StructField("enrollment_status", StringType(), True),
    StructField("source_file_name", StringType(), True),
    StructField("source_received_timestamp", TimestampType(), True),
    StructField("_corrupt_record", StringType(), True),
])

In [0]:
def read_bronze_enrollments(path: str) -> DataFrame:

    return (
        spark.read.format("csv")
        .schema(enrollment_schema)
        .option("header", "true")
        .option("mode", "PERMISSIVE")
        .option("columnNameOfCorruptRecord", "_corrupt_record")
        .load(path)
        .select(
            "*",
            F.col("_metadata.file_path").alias("_source_file_path"),
        )
    )

bronze_df = read_bronze_enrollments(BRONZE_PATH)

records_read = bronze_df.count()

print(f"Enrollment Bronze records read: {records_read:,}")

display(bronze_df.limit(10))

In [0]:
validated_df = (
    bronze_df
    .withColumn(
        "validation_error",
        F.when(
            F.col("enrollment_id").isNull(),
            "Missing enrollment_id"
        )
        .when(
            F.col("employee_id").isNull(),
            "Missing employee_id"
        )
        .when(
            F.col("employer_id").isNull(),
            "Missing employer_id"
        )
        .when(
            F.col("plan_id").isNull(),
            "Missing plan_id"
        )
        .when(
            F.col("coverage_start_date").isNull(),
            "Missing coverage_start_date"
        )
        .when(
            F.col("enrollment_status").isNull(),
            "Missing enrollment_status"
        )
        .when(
            F.col("_corrupt_record").isNotNull(),
            "Corrupt source record"
        )
    )
)

valid_df = (
    validated_df
    .filter(F.col("validation_error").isNull())
)

quarantine_df = (
    validated_df
    .filter(F.col("validation_error").isNotNull())
)

valid_count = valid_df.count()
quarantine_count = quarantine_df.count()

print(f"Valid rows: {valid_count:,}")
print(f"Rejected rows: {quarantine_count:,}")

display(quarantine_df.limit(10))

In [0]:
silver_df = (
    add_audit_columns(
        valid_df,
        batch_id=BATCH_ID,
        source_system=SOURCE_SYSTEM,
    )
)

display(silver_df.limit(10))

In [0]:
(
    silver_df.write
    .format("delta")
    .mode(WRITE_MODE)
    .option("overwriteSchema", "true")
    .save(SILVER_PATH)
)

silver_written_count = (
    spark.read
    .format("delta")
    .load(SILVER_PATH)
    .count()
)

print(f"Silver rows written: {silver_written_count:,}")

In [0]:
quarantine_output_df = quarantine_df.drop("validation_error")

(
    quarantine_output_df.write
    .format("delta")
    .mode(WRITE_MODE)
    .option("overwriteSchema", "true")
    .save(QUARANTINE_PATH)
)

quarantine_written_count = (
    spark.read
    .format("delta")
    .load(QUARANTINE_PATH)
    .count()
)

print(f"Quarantine rows written: {quarantine_written_count:,}")

In [0]:
data_quality_score = calculate_data_quality_score(
    records_read,
    quarantine_count,
)

print_summary(
    records_read=records_read,
    records_written=silver_written_count,
    rejected_records=quarantine_written_count,
    data_quality_score=data_quality_score,
)